In [1]:
import sys
print(sys.executable)   # should end in pharma-rag/.venv/bin/python

/Users/Anneleonore/Documents/githubportfolio/pharma-rag/.venv/bin/python


### 1. Check how many trials (Australia)

In [2]:
import json
import requests
import pandas as pd

BASE = "https://clinicaltrials.gov/api/v2/studies"

r = requests.get(BASE, params={"query.locn": "Australia", "countTotal": "true", "pageSize": 1}, timeout=30)
r.raise_for_status()
print(f"Trials with at least one site in Australia: {r.json()['totalCount']:,}")

Trials with at least one site in Australia: 12,642


### 2. Download sample of 100 trials

In [3]:
r = requests.get(BASE, params={"query.locn": "Australia", "pageSize": 100}, timeout=60)
r.raise_for_status()
studies = r.json()["studies"]

with open("../data/sample_studies.json", "w") as f:
    json.dump(studies, f)

print(f"Downloaded: {len(studies)} trials")
print("\nTop-level sections of one trial:", list(studies[0].keys()))
print("\nModules in protocolSection:")
for module in studies[0]["protocolSection"]:
    print(" -", module)

Downloaded: 100 trials

Top-level sections of one trial: ['protocolSection', 'resultsSection', 'documentSection', 'derivedSection', 'hasResults']

Modules in protocolSection:
 - identificationModule
 - statusModule
 - sponsorCollaboratorsModule
 - oversightModule
 - descriptionModule
 - conditionsModule
 - designModule
 - armsInterventionsModule
 - outcomesModule
 - eligibilityModule
 - contactsLocationsModule
 - referencesModule


### 3. Flatten into table and look at columns

In [12]:
df = pd.json_normalize(studies)
print(f"{df.shape[0]} trials, {df.shape[1]} columns after flattening\n")

# --- 1. Summary per module: how many columns, and how well filled ---
module = [c.split(".")[1] if "." in c else c for c in df.columns]
summary = (pd.DataFrame({"module": module, "filled": df.notna().mean().values})
           .groupby("module")
           .agg(columns=("filled", "size"), avg_filled=("filled", "mean"))
           .sort_values("columns", ascending=False))
summary["avg_filled"] = (summary["avg_filled"] * 100).round(0).astype(int).astype(str) + "%"
print("=== Modules ===")
print(summary.to_string(), "\n")

# --- 2. The fields that matter, with an example value ---
P = "protocolSection."
KEY_FIELDS = {
    "Trial ID":            P + "identificationModule.nctId",
    "Title":               P + "identificationModule.briefTitle",
    "Status":              P + "statusModule.overallStatus",
    "Start date":          P + "statusModule.startDateStruct.date",
    "Study type":          P + "designModule.studyType",
    "Phase":               P + "designModule.phases",
    "Enrolment":           P + "designModule.enrollmentInfo.count",
    "Conditions":          P + "conditionsModule.conditions",
    "Interventions":       P + "armsInterventionsModule.interventions",
    "Lead sponsor":        P + "sponsorCollaboratorsModule.leadSponsor.name",
    "Sponsor type":        P + "sponsorCollaboratorsModule.leadSponsor.class",
    "Has results":         "hasResults",
}

def example(col):
    value = df[col].dropna().iloc[0] if col in df and df[col].notna().any() else "—"
    if isinstance(value, list) and value and isinstance(value[0], dict):   # e.g. interventions
        value = [v.get("name", v) for v in value]
    return str(value)[:70]

rows = [(label,
         f"{df[col].notna().mean():.0%}" if col in df else "missing",
         example(col)) for label, col in KEY_FIELDS.items()]
print("=== Key fields ===")
print(pd.DataFrame(rows, columns=["field", "filled", "example"]).to_string(index=False), "\n")

# --- 3. Text fields for RAG: how long are they? ---
TEXT_FIELDS = {
    "Brief summary":         P + "descriptionModule.briefSummary",
    "Detailed description":  P + "descriptionModule.detailedDescription",
    "Eligibility criteria":  P + "eligibilityModule.eligibilityCriteria",
}
rows = []
for label, col in TEXT_FIELDS.items():
    words = df[col].dropna().str.split().str.len() if col in df else pd.Series(dtype=float)
    rows.append((label, f"{df[col].notna().mean():.0%}" if col in df else "missing",
                 int(words.median()) if len(words) else 0, int(words.max()) if len(words) else 0))
print("=== Text fields (number of words) ===")
print(pd.DataFrame(rows, columns=["field", "filled", "median", "max"]).to_string(index=False))

100 trials, 136 columns after flattening

=== Modules ===
                               columns avg_filled
module                                           
statusModule                        28        63%
designModule                        17        50%
moreInfoModule                      10        28%
eligibilityModule                   10        56%
sponsorCollaboratorsModule           9        38%
identificationModule                 8        85%
miscInfoModule                       6        22%
ipdSharingStatementModule            6        27%
adverseEventsModule                  6        36%
baselineCharacteristicsModule        4        34%
oversightModule                      4        56%
participantFlowModule                4        31%
contactsLocationsModule              3        63%
outcomesModule                       3        68%
referencesModule                     3        24%
descriptionModule                    2        74%
annotationModule                     2    

### Select and recode the relevant columns

In [17]:
def _list(value, key="name"):
    """List (of dicts) -> readable text without duplicates, e.g. 'Rociletinib, Placebo'."""
    if not isinstance(value, list):
        return None
    items = [v.get(key) if isinstance(v, dict) else v for v in value]
    items = [str(i) for i in items if i]
    return ", ".join(dict.fromkeys(items)) or None


def _australian_sites(locations):
    """Trial sites in Australia as 'Facility (City)'."""
    if not isinstance(locations, list):
        return None
    sites = [f"{l.get('facility', '?')} ({l.get('city', '?')})"
             for l in locations if l.get("country") == "Australia"]
    return "; ".join(dict.fromkeys(sites)) or None


def _phase(value):
    """['PHASE1', 'PHASE2'] -> 'Phase 1/2'; 'NA' -> None."""
    if not isinstance(value, list):
        return None
    nums = [p.replace("EARLY_PHASE1", "0").replace("PHASE", "") for p in value if p != "NA"]
    return ("Phase " + "/".join(nums)) if nums else None


def make_records(studies: list) -> pd.DataFrame:
    """One tidy row per trial, with only the fields we need."""
    rows = []
    for s in studies:
        p = s.get("protocolSection", {})
        d = s.get("derivedSection", {})
        ident, status = p.get("identificationModule", {}), p.get("statusModule", {})
        design, cond = p.get("designModule", {}), p.get("conditionsModule", {})
        arms, sponsors = p.get("armsInterventionsModule", {}), p.get("sponsorCollaboratorsModule", {})
        rows.append({
            "trial_id":         ident.get("nctId"),
            "title":            ident.get("briefTitle"),
            "status":           status.get("overallStatus"),
            "why_stopped":      status.get("whyStopped"),
            "start_date":       status.get("startDateStruct", {}).get("date"),
            "study_type":       design.get("studyType"),
            "phase":            _phase(design.get("phases")),
            "enrolment":        design.get("enrollmentInfo", {}).get("count"),
            "conditions":       _list(cond.get("conditions")),
            "condition_terms":  _list(d.get("conditionBrowseModule", {}).get("meshes"), "term"),
            "condition_groups": _list(d.get("conditionBrowseModule", {}).get("ancestors"), "term"),
            "interventions":    _list(arms.get("interventions")),
            "intervention_types": _list(arms.get("interventions"), "type"),
            "drug_terms":       _list(d.get("interventionBrowseModule", {}).get("meshes"), "term"),
            "drug_classes":     _list(d.get("interventionBrowseModule", {}).get("ancestors"), "term"),
            "sponsor":          sponsors.get("leadSponsor", {}).get("name"),
            "sponsor_type":     sponsors.get("leadSponsor", {}).get("class"),
            "collaborators":    _list(sponsors.get("collaborators")),
            "australian_sites": _australian_sites(p.get("contactsLocationsModule", {}).get("locations")),
            "summary":          p.get("descriptionModule", {}).get("briefSummary"),
            "has_results":      s.get("hasResults"),
        })
    df = pd.DataFrame(rows)
    df["start_date"] = pd.to_datetime(df["start_date"], format="mixed", errors="coerce")   # '2012-03' or '2012-03-27'
    df["start_year"] = df["start_date"].dt.year.astype("Int64")
    df["url"] = "https://clinicaltrials.gov/study/" + df["trial_id"]
    return df

### Test sample cell

In [18]:
records = make_records(studies)

print(f"{records.shape[0]} trials x {records.shape[1]} columns\n")
print("Filled per column:")
print(records.notna().mean().mul(100).round(0).astype(int).astype(str).add("%").to_string())

records.head(3)

100 trials x 23 columns

Filled per column:
trial_id              100%
title                 100%
status                100%
why_stopped            10%
start_date            100%
study_type            100%
phase                  81%
enrolment             100%
conditions            100%
condition_terms        86%
condition_groups       84%
interventions          96%
intervention_types     96%
drug_terms             69%
drug_classes           43%
sponsor               100%
sponsor_type          100%
collaborators          26%
australian_sites      100%
summary               100%
has_results           100%
start_year            100%
url                   100%


,trial_id,title,status,why_stopped,start_date,study_type,phase,enrolment,conditions,condition_terms,...,drug_terms,drug_classes,sponsor,sponsor_type,collaborators,australian_sites,summary,has_results,start_year,url
0,NCT01526928,"Study to Evaluate Safety, Pharmacokinetics, and Efficacy...",TERMINATED,NaN,2012-03-27,INTERVENTIONAL,Phase 1/2,612,Locally Advanced or Metastatic Non Small Cell Lung Cancer,"Carcinoma, Non-Small-Cell Lung, Neoplasms, Neoplasm Meta...",...,rociletinib,NaN,"Clovis Oncology, Inc.",INDUSTRY,NaN,Chris O'Brien Lifehouse (Camperdown); Peter MacCallum Ca...,"Rociletinib is a novel, potent, small molecule irreversi...",True,2012,https://clinicaltrials.gov/study/NCT01526928
1,NCT01970865,A Study Of PF-06463922 An ALK/ROS1 Inhibitor In Patients...,COMPLETED,NaN,2014-01-08,INTERVENTIONAL,Phase 1/2,364,ALK-positive Non Small Cell Lung Cancer (NSCLC) and ROS1...,"Carcinoma, Non-Small-Cell Lung",...,"lorlatinib, Crizotinib","Piperidines, Heterocyclic Compounds, 1-Ring, Heterocycli...",Pfizer,INDUSTRY,NaN,Chris O'Brien Lifehouse (Sydney); Royal Prince Alfred Ho...,"Phase 1 and 2 trial to study the safety, pharmacokinetic...",True,2014,https://clinicaltrials.gov/study/NCT01970865
2,NCT02592707,Study to Evaluate the Safety and Preliminary Efficacy of...,TERMINATED,Terminated (Due to small number of ongoing patients. Pat...,2017-03-06,INTERVENTIONAL,Phase 1/2,40,Neuroendocrine Tumors,Neuroendocrine Tumors,...,"177Lu-DOTA-JR11, amino-acid, glucose, and electrolyte so...","Autonomic Agents, Peripheral Nervous System Agents, Phys...",Ipsen,INDUSTRY,NaN,"Peter MacCallum Cancer Centre, Molecular Imaging and Tar...",The purpose of this clinical phase I/II study was to inv...,True,2017,https://clinicaltrials.gov/study/NCT02592707


### Download and save all Australia data

In [19]:
import gzip
import time

def download_all(params: dict) -> list:
    """Fetch every trial matching the parameters, 1,000 per page."""
    all_studies, token, page = [], None, 1
    while True:
        query = {**params, "pageSize": 1000}
        if token:
            query["pageToken"] = token
        for attempt in range(3):                       # retry on a temporary error
            try:
                r = requests.get(BASE, params=query, timeout=120)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                print(f"  page {page}: attempt {attempt + 1} failed ({e}), retrying...")
                time.sleep(5)
        else:
            raise RuntimeError(f"Page {page} failed three times; stopped.")

        data = r.json()
        all_studies.extend(data["studies"])
        print(f"Page {page}: {len(data['studies'])} trials (total so far: {len(all_studies):,})")

        token = data.get("nextPageToken")
        if not token:                                  # no more pages
            return all_studies
        page += 1
        time.sleep(0.5)                                # be polite to the server


all_studies = download_all({"query.locn": "Australia"})
print(f"\nDownloaded: {len(all_studies):,} trials")

# Keep a compressed copy of the raw data, so we never have to download again
with gzip.open("../data/trials_australia_raw.json.gz", "wt") as f:
    json.dump(all_studies, f)

# Turn into tidy records with the same function as for the sample, and save
records = make_records(all_studies)
records.to_parquet("../data/trials_australia.parquet", index=False)

print(f"Records: {records.shape[0]:,} trials x {records.shape[1]} columns")
print("Saved: data/trials_australia_raw.json.gz and data/trials_australia.parquet")

Page 1: 1000 trials (total so far: 1,000)
Page 2: 1000 trials (total so far: 2,000)
Page 3: 1000 trials (total so far: 3,000)
Page 4: 1000 trials (total so far: 4,000)
Page 5: 1000 trials (total so far: 5,000)
Page 6: 1000 trials (total so far: 6,000)
Page 7: 1000 trials (total so far: 7,000)
Page 8: 1000 trials (total so far: 8,000)
Page 9: 1000 trials (total so far: 9,000)
Page 10: 1000 trials (total so far: 10,000)
Page 11: 1000 trials (total so far: 11,000)
Page 12: 1000 trials (total so far: 12,000)
Page 13: 642 trials (total so far: 12,642)

Downloaded: 12,642 trials
Records: 12,642 trials x 23 columns
Saved: data/trials_australia_raw.json.gz and data/trials_australia.parquet


In [20]:
print(f"Unique trials: {records['trial_id'].nunique():,} of {len(records):,}")
print(records.notna().mean().mul(100).round(0).astype(int).astype(str).add("%").to_string())

Unique trials: 12,642 of 12,642
trial_id              100%
title                 100%
status                100%
why_stopped            11%
start_date            100%
study_type            100%
phase                  80%
enrolment              99%
conditions            100%
condition_terms        86%
condition_groups       85%
interventions          96%
intervention_types     96%
drug_terms             64%
drug_classes           40%
sponsor               100%
sponsor_type          100%
collaborators          29%
australian_sites      100%
summary               100%
has_results           100%
start_year            100%
url                   100%
